<a href="https://colab.research.google.com/github/embark-cybertraining/embark-scratch-notebooks/blob/main/session_3/session_3_EDI_OBIS_map_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EDI_OBIS Use Case Goal

The goal of this workflow is to demonstrate interoperable access, inspection, and visualization of environmental and biodiversity data from multiple marine data repositories within the California Current Large Marine Ecosystem.

Additional code at end to investigate dataset bounding boxes and the combined boxes, and querying obis by larger polygon instead of area_id for california current

## More details about workflow and datasets

The workflow combines:

- particulate organic matter sampling data from the California Current Ecosystem Long-Term Ecological Research (CCE-LTER) program hosted by EDI
- marine species occurrence records from OBIS

Usecase info here: https://docs.google.com/document/d/1bKVoiHk6WYLtEDKojf4Rn7LIS-R9YAom/edit#heading=h.jx14u33ubg8w


# Setup

Install needed packages

In [ ]:
%pip install folium
%pip install geopandas
%pip install pyobis
%pip install shapely

# Dataset 1: EDI

**Repository:** Environmental Data Initiative (EDI)  
**Dataset landing page:**  
https://portal.edirepository.org/nis/mapbrowse?packageid=knb-lter-cce.104.13

**Direct data access endpoint used in notebook:**  
https://pasta.lternet.edu/package/data/eml/knb-lter-cce/104/13/01cc01e9b988aea447b1dd417eca7938

### EDI Dataset Summary

This dataset contains particulate organic matter (POM) measurements collected by the California Current Ecosystem Long-Term Ecological Research (CCE-LTER) program.

The notebook uses:
- sampling coordinates
- sampling depth
- date/time
- cruise and station metadata
- particulate carbon and nitrogen measurements

These data provide environmental and biogeochemical context for comparison with marine biodiversity occurrence records.

In [ ]:
# Setup/imports

import requests, re, folium, html
import geopandas as gpd #for bounding box things
from shapely.geometry import box
from shapely import wkt as shapely_wkt
import pandas as pd
from pyobis import occurrences
from IPython.display import display #used for better df display in notebook
from IPython.display import HTML #used for clickable links in species list table

SCI_NAME = 'Pyrosoma' # second run we will use 'Thaliacea'
#Note: we are going to start by running this notebook with SCI_NAME 'Pyrosoma'
#   then broadening to family 'Thaliacea' in another notebook run.

# In OBIS this area_id corresponds to Marine Region for California Current 8549
AREA_ID = 40003 #OBIS area_id
MRGID = 8549 #Marine Regions MRGID

RECORD_LIMIT = 10000 # The record limit we will use with our OBIS request.
# https://obis.org/data/access/ has more information about small
#    vs large request strategy.
# We use the record limit with checks after we get the data to make sure
#    we got the entire dataset.

## Access Data: EDI URL

strategy: Load from a file using URL and access key

In [ ]:
# Access EDI data
# Requirement: You have obtained an EDI access key and have it acccessible to this code.
#   * In google colab you will have entered a "Secret" for the EDI_API_KEY

base_url = (
    "https://pasta.lternet.edu/package/data/eml/"
    "knb-lter-cce/104/13/01cc01e9b988aea447b1dd417eca7938"
)

url = base_url

try: #this is how the data access will work if you use Google Colab
    from google.colab import userdata

    EDI_API_KEY = userdata.get("EDI_ACCESS_KEY")

    if EDI_API_KEY:
        url = f"{base_url}?key={EDI_API_KEY}"
        print("Using EDI access key")
    else:
        print("No EDI access key found, using public access")

except ImportError:
    print("Not running in Google Colab, using public access")

df_edi = pd.read_csv(url)

print(f"EDI records retrieved: {len(df_edi)}")

print("Show random 5 rows:")
df_edi.sample(5)

Using EDI access key
EDI records retrieved: 3295
Show random 5 rows:


,studyName,Datetime UTC,Transect,Station,Latitude (º),Longitude (º),R2R Event,CCE Event,Cast Number,Sample Type,Bottle Number,Depth (m),N (µg/L),C (µg/L),N (µmol/L),C (µmol/L),C/N Molar Ratio,Notes,Cruise,Associative Bottle Number
34,P0605KN,2006-05-15 09:08:45,CYCLE 1,NaN,34.259540,-121.056800,NaN,219.0,19,NaN,20.0,8.0,89.391136,463.051331,6.382009,38.553234,6.040925,NaN,CCE-P0605,NaN
1487,P1408MV,2014-08-12 17:53:23,CYCLE 1,NaN,34.761280,-121.081100,NaN,118.0,9,NaN,11.0,38.0,38.971346,203.365254,2.782328,16.932007,6.085554,NaN,CCE-P1408,NaN
1391,P1208MV,2012-08-20 23:38:50,EFRONT2,NaN,34.557580,-122.521200,NaN,503.0,58,NaN,5.0,22.0,53.080898,296.996613,3.789668,24.727669,6.525022,NaN,CCE-P1208,NaN
2843,P2107,2021-07-15 15:58:49,Radiator Survey 1,S1-10,35.417392,-121.416971,20210715.1558.001,25.0,910,Underway surface samples,NaN,4.0,93.441755,625.326063,6.671199,52.064081,7.804306,NaN,CCE-P2107,NaN
213,P0704TN,2007-04-05 09:43:07,Cycle 1,NaN,34.277000,-120.911000,NaN,75.0,8,NaN,2.0,50.0,5.304607,32.424839,0.378718,2.699663,7.128421,NaN,CCE-P0704,NaN


In [ ]:
df_edi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3295 entries, 0 to 3294
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   studyName                  3295 non-null   object 
 1   Datetime UTC               3295 non-null   object 
 2   Transect                   3199 non-null   object 
 3   Station                    468 non-null    object 
 4   Latitude (º)               3278 non-null   float64
 5   Longitude (º)              3278 non-null   float64
 6   R2R Event                  468 non-null    object 
 7   CCE Event                  3081 non-null   float64
 8   Cast Number                3295 non-null   int64  
 9   Sample Type                468 non-null    object 
 10  Bottle Number              3238 non-null   float64
 11  Depth (m)                  3238 non-null   float64
 12  N (µg/L)                   3295 non-null   float64
 13  C (µg/L)                   3295 non-null   float

## Quality Control and Cleaning

In [ ]:
# Transform EDI data
#  to use Darwin Core names for lat and lon
# Remove rows with no lat/lon
# Remove rows with missing varibles of interest (particulate C or N values)

# variables of interest
vars_of_interest = [
    "N (µg/L)",
    "C (µg/L)"
]

print(f"EDI records retrieved (total): {len(df_edi)}")

# Rename lat/lon columns (to Darwin Core standard)
#   and clean missing coordinates
df_edi = df_edi.rename(columns={
    "Longitude (º)": "decimalLongitude",
    "Latitude (º)": "decimalLatitude"
})

df_edi[["decimalLatitude", "decimalLongitude"]] = (
    df_edi[["decimalLatitude", "decimalLongitude"]]
    .apply(pd.to_numeric, errors="coerce")
)

print("Rows removed - missing [lat,lon]:",
      df_edi[["decimalLatitude", "decimalLongitude"]]
      .isna().any(axis=1).sum())

df_edi = df_edi.dropna(
    subset=["decimalLatitude", "decimalLongitude"]
)
# ---- Vars of interest

print("Rows removed - missing "+str(vars_of_interest)+":",
      df_edi[vars_of_interest]
      .isna()
      .all(axis=1)
      .sum()
    )

# Drop rows where all target variables are missing
df_edi = df_edi.dropna(
        subset=vars_of_interest,
        how="all"
    )

print(f"Rows remaining: {len(df_edi)}")

EDI records retrieved (total): 3295
Rows removed - missing [lat,lon]: 17
Rows removed - missing ['N (µg/L)', 'C (µg/L)']: 0
Rows remaining: 3278


In [ ]:
# Inspect EDI data

# do this todo clean where no target variable we re interesed in (drop if no C or N)

print("\nDatetime range:")
print(pd.to_datetime(df_edi["Datetime UTC"], errors="coerce").min())
print(pd.to_datetime(df_edi["Datetime UTC"], errors="coerce").max())

print("\nDepth range:")
print(pd.to_numeric(df_edi["Depth (m)"], errors="coerce").min())
print(pd.to_numeric(df_edi["Depth (m)"], errors="coerce").max())

## todo: add min/max of variables particulate C&N of interest


Datetime range:
2006-05-11 09:40:48
2024-03-17 10:08:00

Depth range:
1.0
801.0


## Map EDI Data


In [ ]:
# Create reusable feature group for EDI points
edi_popup_cols = df_edi.columns.tolist()

edi_points = folium.FeatureGroup(name="EDI Points")

# this section formats information to show in a popup when you click the point
for _, row in df_edi.iterrows():
    popup_rows = [
        f"<tr><th>{col}</th><td>{row[col]}</td></tr>"
        for col in edi_popup_cols
        if pd.notna(row.get(col))
    ]

    popup = folium.Popup(
        f"<table>{''.join(popup_rows)}</table>",
        max_width=300
    )

    folium.CircleMarker(
        location=[row["decimalLatitude"], row["decimalLongitude"]],
        radius=5,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.7,
        popup=popup,
    ).add_to(edi_points)


# make a map
m_cc = folium.Map(
    location=[37, -123],
    zoom_start=5,
    tiles="OpenStreetMap"
)

# add points
edi_points.add_to(m_cc)

m_cc

# Dataset 2: OBIS

## OBIS Dataset Summary

**Repository:** Ocean Biodiversity Information System (OBIS)  
**API endpoint used in notebook:**  
https://api.obis.org/v3/occurrence

**OBIS area reference:**  
https://api.obis.org/v3/area

**OBIS source information:**  
https://obis.org/sources/

### OBIS Query Summary

The notebook queries OBIS occurrence records for:

- scientific name: *Pyrosoma atlanticum*
- region: California Current Large Marine Ecosystem

## Access Data: OBIS API

Strategy: Access data using the OBIS API directly (not requiring an additional library).

In [ ]:
# Access OBIS data as DataFrame

response = requests.get(
    "https://api.obis.org/v3/occurrence",
    params={
        "scientificname": SCI_NAME,
        "areaid": AREA_ID,
        "size": RECORD_LIMIT,
    },
)

df_obis = pd.DataFrame(response.json()["results"])

print(f"OBIS records retrieved: {len(df_obis)}")

if len(df_obis) < RECORD_LIMIT:
    print("All matching records retrieved.")
else:
    print("Record limit reached.")



OBIS records retrieved: 1870
All matching records retrieved.


In [ ]:
df_obis.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1870 entries, 0 to 1869
Columns: 131 entries, basisOfRecord to habitat
dtypes: bool(3), float64(15), int64(11), object(102)
memory usage: 1.8+ MB


In [ ]:
df_obis.columns.to_list()

['basisOfRecord',
 'class',
 'classid',
 'datasetID',
 'date_end',
 'date_mid',
 'date_start',
 'date_year',
 'decimalLatitude',
 'decimalLongitude',
 'depth',
 'eventDate',
 'eventID',
 'family',
 'familyid',
 'genus',
 'genusid',
 'geodeticDatum',
 'higherClassification',
 'identificationRemarks',
 'institutionID',
 'kingdom',
 'kingdomid',
 'license',
 'locality',
 'locationID',
 'marine',
 'materialSampleID',
 'maximumDepthInMeters',
 'minimumDepthInMeters',
 'occurrenceID',
 'occurrenceStatus',
 'order',
 'orderid',
 'organismQuantity',
 'organismQuantityType',
 'parentEventID',
 'phylum',
 'phylumid',
 'recordedBy',
 'sampleSizeUnit',
 'sampleSizeValue',
 'scientificName',
 'scientificNameID',
 'species',
 'speciesid',
 'subfamily',
 'subfamilyid',
 'subphylum',
 'subphylumid',
 'taxonRank',
 'type',
 'verbatimIdentification',
 'waterBody',
 'id',
 'dataset_id',
 'node_id',
 'dropped',
 'absence',
 'originalScientificName',
 'aphiaID',
 'flags',
 'bathymetry',
 'shoredistance',
 

## Quality Control and Cleaning

In [ ]:
# Clean, inspect, and summarize OBIS data

#Not all records show the datasetName or datasetID but that can be obtained
# from the obis identifier dataset_id using another query
def get_obis_dataset_title(dataset_id):
    try:
        r = requests.get(
            "https://api.obis.org/dataset",
            params={"datasetid": dataset_id},
            timeout=30,
        )
        r.raise_for_status()

        data = r.json()

        if isinstance(data, dict):
            records = data.get("results", data.get("data", []))
        else:
            records = data

        if records:
            return records[0].get("title", "")

    except Exception:
        pass

    return ""


# Add dataset titles from OBIS dataset API
dataset_titles = {
    dataset_id: get_obis_dataset_title(dataset_id)
    for dataset_id in df_obis["dataset_id"].dropna().unique()
}

df_obis["datasetTitle"] = df_obis["dataset_id"].map(dataset_titles)


df_obis["decimalLatitude"] = pd.to_numeric(
    df_obis["decimalLatitude"],
    errors="coerce"
)

df_obis["decimalLongitude"] = pd.to_numeric(
    df_obis["decimalLongitude"],
    errors="coerce"
)

# We only want records where there are lat/lons
df_obis = df_obis.dropna(
    subset=["decimalLatitude", "decimalLongitude"]
).copy()

preview_cols = [ #used for displaying df and in map popup when clicking point
    "scientificName",
    "scientificNameID",  # LSID
    "aphiaID",
    "decimalLatitude",
    "decimalLongitude",
    "eventDate",
    "minimumDepthInMeters",
    "maximumDepthInMeters",
    "occurrenceStatus",
    "datasetTitle",
    "occurrenceID",
]

preview_cols = [c for c in preview_cols if c in df_obis.columns]

# Display what we got
print("\nPreview of key fields (random 10 rows):")
display(df_obis[preview_cols].sample(10))




Preview of key fields (random 10 rows):


,scientificName,scientificNameID,aphiaID,decimalLatitude,decimalLongitude,eventDate,minimumDepthInMeters,maximumDepthInMeters,occurrenceStatus,datasetTitle,occurrenceID
1166,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,34.407324,-119.549823,2021-07-07T19:16:05-07:00,NaN,NaN,urn:lsid:marinespecies.org:taxname:137250,iNaturalist Research-grade Observations Marine...,https://www.inaturalist.org/observations/86115650
1340,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,34.458383,-120.024615,2023-06-21T14:41:13-07:00,NaN,NaN,urn:lsid:marinespecies.org:taxname:137250,iNaturalist Research-grade Observations Marine...,https://www.inaturalist.org/observations/16890...
757,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,38.465700,-123.386000,2014-06-11T06:11:15Z,25.00,30.00,present,Rockfish Recruitment and Ecosystem Assessment ...,2950def0-e4eb-48f9-a974-8630b8bae81e
908,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,34.409237,-119.869562,2021-07-02T16:34:43-07:00,NaN,NaN,urn:lsid:marinespecies.org:taxname:137250,iNaturalist Research-grade Observations Marine...,https://www.inaturalist.org/observations/85583165
1853,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,36.996700,-122.306700,2001-06-02T04:18:00Z,25.00,30.00,present,Rockfish Recruitment and Ecosystem Assessment ...,8894aedc-d9fe-4432-b16a-560deed65c96
1182,Pyrosoma,urn:lsid:marinespecies.org:taxname:137224,137224,36.588810,-122.518013,2000-11-15T06:33:00Z,29.37,29.37,NaN,Video Annotation and Reference System (VARS) d...,NaN
1392,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,34.424439,-119.906522,2020-11-26T13:12:45Z,NaN,NaN,urn:lsid:marinespecies.org:taxname:137250,iNaturalist Research-grade Observations Marine...,https://www.inaturalist.org/observations/65663297
529,Pyrosoma,urn:lsid:marinespecies.org:taxname:137224,137224,36.720425,-122.064590,2000-09-27T07:34:59Z,75.10,75.10,NaN,Video Annotation and Reference System (VARS) d...,NaN
618,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,36.554684,-121.930871,2025-01-11T11:35:55-08:00,NaN,NaN,urn:lsid:marinespecies.org:taxname:137250,iNaturalist Research-grade Observations Marine...,https://www.inaturalist.org/observations/25816...
172,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,36.983800,-122.753200,2014-05-19T11:34:36Z,25.00,30.00,present,Rockfish Recruitment and Ecosystem Assessment ...,40ff13e1-ba17-4aef-a781-bad36a415753


In [ ]:
# Further Inspect data (data sources, ids, etc)

# Unique species with WoRMS page links
species_list_df = (
    df_obis[["scientificName", "scientificNameID", "aphiaID"]]
    .drop_duplicates()
    .sort_values(by="scientificName")
    .reset_index(drop=True)
)

species_list_df["WoRMS_link"] = species_list_df["aphiaID"].apply(
    lambda x: (
        f'<a href="https://marinespecies.org/aphia.php?p=taxdetails&id={int(x)}" '
        f'target="_blank">WoRMS {int(x)}</a>'
    )
    if pd.notna(x)
    else ""
)

# Count occurrenceStatus values with associated scientific names

occurrence_status_summary = (
    df_obis.groupby("occurrenceStatus")["scientificName"]
    .agg(
        count="size",
        unique_scientificNames=lambda x: sorted(x.dropna().unique())
    )
    .reset_index()
)

# Show unique occurrenceStatus values for each scientificName

species_occ_status = (
    df_obis[
        ["scientificName", "occurrenceStatus"]
    ]
    .fillna("Not Supplied")
    .value_counts()
    .reset_index(name="count")
    .reset_index(drop=True)
)

# Unique datasets with clickable OBIS dataset links
df_obis["datasetURL"] = (
    "https://obis.org/dataset/" + df_obis["dataset_id"].astype(str)
)

datasets = (
    df_obis[["dataset_id", "datasetTitle", "datasetURL"]]
    .drop_duplicates()
    .sort_values(by="datasetTitle")
    .reset_index(drop=True)
)

# Make datasetURL clickable
datasets["datasetURL"] = datasets["datasetURL"].apply(
    lambda x: f'<a href="{x}" target="_blank">{x}</a>'
)


# Display more info

print(f"\nTotal unique datasets: {len(datasets)}")
display(HTML(datasets.to_html(escape=False, index=False)))

print("Unique species in the dataset:")
display(HTML(species_list_df.to_html(escape=False, index=False)))

print("Unique occurrenceStatus values per scientificName:")
display(species_occ_status)

print("\nDataFrame cols")
print(df_obis.columns.tolist())



Total unique datasets: 8


dataset_id,datasetTitle,datasetURL
aa16d305-d413-4c4a-90be-b1ec3298d58d,CAS Invertebrate Zoology (IZ),https://obis.org/dataset/aa16d305-d413-4c4a-90be-b1ec3298d58d
92af50ad-c9d7-4fe7-b012-eae0e77233a2,Los Angeles Urban Ocean Expedition 2019,https://obis.org/dataset/92af50ad-c9d7-4fe7-b012-eae0e77233a2
71c2c816-7e94-40b9-8e28-8172d9c5fefb,NOAA Ocean Exploration seawater eDNA metabarcoding (EX2301),https://obis.org/dataset/71c2c816-7e94-40b9-8e28-8172d9c5fefb
89e23fc8-3f61-4480-9de3-358fe6eefe0b,National Museum of Natural History Invertebrate Zoology Collections,https://obis.org/dataset/89e23fc8-3f61-4480-9de3-358fe6eefe0b
5b6251f6-a7a5-4dc9-994d-c9504b54776f,"Rockfish Recruitment and Ecosystem Assessment Survey, Catch Data",https://obis.org/dataset/5b6251f6-a7a5-4dc9-994d-c9504b54776f
c5687a17-e454-40f9-9a4b-d04b2c812d74,UF Invertebrate Zoology,https://obis.org/dataset/c5687a17-e454-40f9-9a4b-d04b2c812d74
a419c8da-35ed-4b62-9709-39b56369c44e,Video Annotation and Reference System (VARS) database,https://obis.org/dataset/a419c8da-35ed-4b62-9709-39b56369c44e
eaea291a-1e1d-4382-b86f-ac3cc15b8d5a,iNaturalist Research-grade Observations Marine Subset,https://obis.org/dataset/eaea291a-1e1d-4382-b86f-ac3cc15b8d5a


Unique species in the dataset:


scientificName,scientificNameID,aphiaID,WoRMS_link
Pyrosoma,urn:lsid:marinespecies.org:taxname:137224,137224,WoRMS 137224
Pyrosoma,NaN,137224,WoRMS 137224
Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,WoRMS 137250
Pyrosoma atlanticum,NaN,137250,WoRMS 137250


Unique occurrenceStatus values per scientificName:


,scientificName,occurrenceStatus,count
0,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,1260
1,Pyrosoma atlanticum,present,477
2,Pyrosoma,Not Supplied,127
3,Pyrosoma atlanticum,Not Supplied,6



DataFrame cols
['basisOfRecord', 'class', 'classid', 'datasetID', 'date_end', 'date_mid', 'date_start', 'date_year', 'decimalLatitude', 'decimalLongitude', 'depth', 'eventDate', 'eventID', 'family', 'familyid', 'genus', 'genusid', 'geodeticDatum', 'higherClassification', 'identificationRemarks', 'institutionID', 'kingdom', 'kingdomid', 'license', 'locality', 'locationID', 'marine', 'materialSampleID', 'maximumDepthInMeters', 'minimumDepthInMeters', 'occurrenceID', 'occurrenceStatus', 'order', 'orderid', 'organismQuantity', 'organismQuantityType', 'parentEventID', 'phylum', 'phylumid', 'recordedBy', 'sampleSizeUnit', 'sampleSizeValue', 'scientificName', 'scientificNameID', 'species', 'speciesid', 'subfamily', 'subfamilyid', 'subphylum', 'subphylumid', 'taxonRank', 'type', 'verbatimIdentification', 'waterBody', 'id', 'dataset_id', 'node_id', 'dropped', 'absence', 'originalScientificName', 'aphiaID', 'flags', 'bathymetry', 'shoredistance', 'sst', 'sss', 'country', 'datasetName', 'eventRe

# Map EDI and OBIS together

Plot points from EDI and OBIS data together on one map.

In [ ]:

# Use only preview columns in popup
obis_popup_cols = preview_cols

# Create reusable feature group for OBIS points
obis_points = folium.FeatureGroup(name="OBIS Points")

#this section formats information to show in popups when you click a point
for _, row in df_obis.iterrows():
    popup_rows = [
        f"<tr><th>{col}</th><td>{row[col]}</td></tr>"
        for col in obis_popup_cols
        if col in df_obis.columns and pd.notna(row.get(col))
    ]

    popup = folium.Popup(
        f"<table>{''.join(popup_rows)}</table>",
        max_width=300
    )

    folium.CircleMarker(
        location=[row["decimalLatitude"], row["decimalLongitude"]],
        radius=5,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.7,
        popup=popup,
    ).add_to(obis_points)


In [ ]:
# Map EDI and OBIS data together

#map m_cc was already created in the EDI data section.

# add new OBIS points to the map that already had EDI points
obis_points.add_to(m_cc)

m_cc

# Additional Access Strategy 1: pyobis library

Load data from a bounding box using the pyobis library.  We will calculate the bounding box of the combined EDI and OBIS data. Then we will use that bounding box in a new query to get OBIS data using the python libary pyobis.

## Calculate combined data bounding box

In [ ]:
# Get data bounding box (WKT) from superset of EDI and OBIS data
# we did this above, but this cell will need
#   from shapely.geometry import box
#   import pandas as pd

# Combine lat/lon from both datasets and calculate one bounding box directly
combined_lats = pd.concat([df_obis["decimalLatitude"], df_edi["decimalLatitude"]])
combined_lons = pd.concat([df_obis["decimalLongitude"], df_edi["decimalLongitude"]])

minx, maxx = combined_lons.min(), combined_lons.max()
miny, maxy = combined_lats.min(), combined_lats.max()

# Build the combined bounding box polygon and WKT
combined_bbox_polygon = box(minx, miny, maxx, maxy)
combined_bbox_wkt = combined_bbox_polygon.wkt

print("Combined bounding box WKT:")
print(combined_bbox_wkt)

Combined bounding box WKT:
POLYGON ((-116.932998657 29.8670005798, -116.932998657 46.5919961817, -130.547487 46.5919961817, -130.547487 29.8670005798, -116.932998657 29.8670005798))


In [ ]:
# Create map centered on combined bounding box
m_combined = folium.Map(
    location=[
        (combined_bbox_polygon.bounds[1] + combined_bbox_polygon.bounds[3]) / 2,
        (combined_bbox_polygon.bounds[0] + combined_bbox_polygon.bounds[2]) / 2
    ],
    zoom_start=5,
    tiles="OpenStreetMap"
)

# Add points from before
edi_points.add_to(m_combined)
obis_points.add_to(m_combined)


# Add combined bounding box with WKT popup
folium.GeoJson(
    combined_bbox_polygon,
    name="Combined Bounding Box",
    style_function=lambda x: {
        "color": "purple",
        "weight": 3,
        "fill": False
    },
    popup=folium.Popup(combined_bbox_polygon.wkt, max_width=500)
).add_to(m_combined)

# Add layer control
folium.LayerControl().add_to(m_combined)

# Add popup showing latitude/longitude anywhere clicked on the map
m_combined.add_child(folium.LatLngPopup())

m_combined

# Looking at expanded BB for OBIS data

query using pyobis and the polyon of the previous obis query which corresponded to data from  area 40003 plus edi dataset bounds

In [ ]:
# Access OBIS data using pyobis and the combined bounding box
#   * We imported at the top of our notebook, but this cell needs:
#     from pyobis import occurrences

response = occurrences.search(
    scientificname=SCI_NAME,
    geometry=combined_bbox_wkt, #calculated from our edi and obis data
    size=RECORD_LIMIT
)

df_obis_expanded = response.execute()


print(f"OBIS records retrieved: {len(df_obis_expanded)}")

if len(df_obis_expanded) < RECORD_LIMIT:
    print("All matching records retrieved.")
else:
    print("Record limit reached.")



OBIS records retrieved: 2040
All matching records retrieved.


In [ ]:
# Find OBIS expanded records that were not present in the original dataset

# Use occurrenceID if available and unique
new_obis_records = df_obis_expanded[
    ~df_obis_expanded["occurrenceID"].isin(df_obis["occurrenceID"])
].copy()

print(f"New OBIS records found: {len(new_obis_records)}")

# Preview new records
new_obis_records.head()

New OBIS records found: 168


,basisOfRecord,catalogNumber,class,classid,collectionCode,coordinateUncertaintyInMeters,countryCode,datasetName,dateIdentified,date_end,...,georeferenceRemarks,nomenclaturalCode,unaccepted,verbatimLatitude,verbatimLongitude,footprintSRS,startDayOfYear,georeferenceSources,verbatimCoordinates,habitat
0,HumanObservation,78006997,Thaliacea,22626,Observations,90.0,US,iNaturalist research-grade observations,2021-05-09T04:12:44Z,1.620432e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,HumanObservation,84914478,Thaliacea,22626,Observations,5.0,US,iNaturalist research-grade observations,2021-06-30T22:30:43Z,1.624838e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,HumanObservation,184442589,Thaliacea,22626,Observations,150.0,US,iNaturalist research-grade observations,2023-09-22T21:41:45Z,1.626221e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,HumanObservation,105248276,Thaliacea,22626,Observations,4.0,US,iNaturalist research-grade observations,2022-01-20T04:56:32Z,1.642464e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
39,HumanObservation,172415545,Thaliacea,22626,Observations,4.0,US,iNaturalist research-grade observations,2023-07-12T02:51:41Z,1.689034e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Stop notebook execution if no new OBIS records were found
if len(new_obis_records) == 0:
    raise SystemExit(
        f"No new OBIS records found for {SCI_NAME}. Stopping notebook execution."
    )

In [ ]:
# Rebuild combined map with new points included

m_combined_with_new = folium.Map(
    location=[
        new_obis_records["decimalLatitude"].mean(),
        new_obis_records["decimalLongitude"].mean()
    ],
    zoom_start=5,
    tiles="OpenStreetMap"
)

# Add existing reusable layers
edi_points.add_to(m_combined_with_new)
obis_points.add_to(m_combined_with_new)

# Create reusable layer for new OBIS points
new_obis_points = folium.FeatureGroup(
    name="New OBIS Points",
    show=True
)

# Add new OBIS points to the reusable layer
for _, row in new_obis_records.dropna(
    subset=["decimalLatitude", "decimalLongitude"]
).iterrows():

    popup_rows = [
        f"<tr><th>{col}</th><td>{row[col]}</td></tr>"
        for col in preview_cols
        if col in new_obis_records.columns and pd.notna(row.get(col))
    ]

    popup = folium.Popup(
        f"<table>{''.join(popup_rows)}</table>",
        max_width=300
    )

    folium.CircleMarker(
        location=[row["decimalLatitude"], row["decimalLongitude"]],
        radius=8,
        color="orange",
        fill=True,
        fill_color="orange",
        fill_opacity=1,
        popup=popup,
    ).add_to(new_obis_points)

# Add new-points layer to map
new_obis_points.add_to(m_combined_with_new)

# Zoom to new points
m_combined_with_new.fit_bounds([
    [
        new_obis_records["decimalLatitude"].min(),
        new_obis_records["decimalLongitude"].min()
    ],
    [
        new_obis_records["decimalLatitude"].max(),
        new_obis_records["decimalLongitude"].max()
    ]
])

#show the bounding box used for the full obis query (using pyobis)
folium.GeoJson(
    combined_bbox_polygon,
    name="Combined Bounding Box",
    style_function=lambda x: {
        "color": "purple",
        "weight": 3,
        "fill": False
    },
    popup=folium.Popup(
        combined_bbox_polygon.wkt,
        max_width=500
    )
).add_to(m_combined_with_new)

# Add layer control after ALL layers are added
folium.LayerControl(collapsed=False).add_to(
    m_combined_with_new
)

m_combined_with_new

In [ ]:
# Save Folium map as HTML
#  You can open this html file to view it with any web browser
m_combined_with_new.save(SCI_NAME+"-combined_map_with_bb.html")

# Additional Access Strategy: Marine Regions API

Exploring the Marine Regions API

Previously we made our own geometries and mapped them. However, now we can try pulling geometries from the Marine Regions (marineregions.org) API.

https://www.marineregions.org/gazetteer.php?id=8549&p=details

In [ ]:
#At the top of the notebook we defined the region id
#  MRGID = 8549  # California Current (Marine Regions)
#      corresponding to OBIS area id 40003

url = f"https://www.marineregions.org/rest/getGazetteerGeometries.jsonld/{MRGID}/"
r = requests.get(url, timeout=60)
r.raise_for_status()

data = r.json()
geometry_wkt = None

for geom in data.get("mr:hasGeometry", []):
    wkt = geom.get("gsp:asWKT")
    if wkt:
        geometry_wkt = re.sub(r"<[^>]+>\s*", "", wkt).strip()
        break

print("Preview the geom")
print(geometry_wkt[0:100])
print("Geometry length="+str(len(geometry_wkt)))

Preview the geom
POLYGON ((-109.91147613 22.8783226, -108.75467682 21.83747101, -108.7578125 21.83586121, -109.616462
Geometry length=108685


In [ ]:
from shapely import wkt as shapely_wkt
from shapely.geometry import mapping
from shapely.validation import make_valid

# Parse the Marine Regions boundary and repair it if invalid
marine_region_geom = shapely_wkt.loads(geometry_wkt)
if not marine_region_geom.is_valid:
   print('fixing marine region geometry')
   marine_region_geom = make_valid(marine_region_geom)

region_geojson = {
    "type": "Feature",
    "properties": {"MRGID": MRGID},
    "geometry": mapping(marine_region_geom)
}

In [ ]:
# Combine the extent of both point datasets and the region boundary
region_minx, region_miny, region_maxx, region_maxy = marine_region_geom.bounds
all_lats = pd.concat([df_edi["decimalLatitude"], df_obis["decimalLatitude"],
                       pd.Series([region_miny, region_maxy])])
all_lons = pd.concat([df_edi["decimalLongitude"], df_obis["decimalLongitude"],
                       pd.Series([region_minx, region_maxx])])

# Build a fresh map with the EDI points, OBIS points, and region boundary
m_final = folium.Map(tiles="OpenStreetMap")

edi_points.add_to(m_final)
obis_points.add_to(m_final)
new_obis_points.add_to(m_final)

# Add combined bounding box
folium.GeoJson(
    combined_bbox_polygon,
    name="Combined Bounding Box",
    style_function=lambda x: {
        "color": "purple",
        "weight": 3,
        "fill": False
    },
    popup=folium.Popup(combined_bbox_polygon.wkt, max_width=500)
).add_to(m_final)

#Marine region geometry
folium.GeoJson(
    region_geojson,
    name=f"Marine Region Boundary (MRGID {MRGID})",
    style_function=lambda x: {
        "color": "green",
        "weight": 3,
        "fill": False#,
        #"dashArray": "5, 5"
    },
    popup=folium.Popup(f"Marine Region MRGID {MRGID}", max_width=300)
).add_to(m_final)

folium.LayerControl(collapsed=False).add_to(m_final)

m_final

In [ ]:
# Save the final map as HTML
#  You can open this html file to view it with any web browser
m_final.save("session_3_"+SCI_NAME + "-final_map.html")